# Training for PropLenght

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import os
from helper.pipeline_utils import RegressionDataset, Net, compute_loss, compute_nth_r2


def run_pipeline(
    TRAIN_PATH: str,
    BATCH_SIZE: int,
    LR: float,
    NUM_EPOCHS: int,
    WEIGHT_DECAY: float,
    NOISE_STD: float,
    SEED: int,
    hidden_dims: list,
    activation: str,
    output_weights: list,
    use_validation: bool,
    validation_split: float,
    save_artifacts: bool = True,
):
    """
    Training pipeline with optional artifact saving (models & params only),
    now using compute_nth_r2 for the first output (index=0) instead of compute_first_r2.
    """
    # [1] Setup & reproducibility
    tag = uuid.uuid4().hex[:8]
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # [2] Load & optionally split data
    df = pd.read_csv(TRAIN_PATH)
    X_all = df.iloc[:, :14].values
    Y_all = df.iloc[:, 14:].values

    if use_validation:
        X_train, X_val, Y_train, Y_val = train_test_split(
            X_all, Y_all, test_size=validation_split, random_state=SEED
        )
    else:
        X_train, Y_train = X_all, Y_all
        X_val = Y_val = None

    # [3] Feature scaling (fit on train only)
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(Y_train)

    X_train = x_scaler.transform(X_train)
    Y_train = y_scaler.transform(Y_train)
    if use_validation:
        X_val = x_scaler.transform(X_val)
        Y_val = y_scaler.transform(Y_val)

    # [4] Prepare datasets & loaders
    train_ds_aug   = RegressionDataset(X_train, Y_train, augment=True,  noise_std=NOISE_STD)
    train_ds_clean = RegressionDataset(X_train, Y_train, augment=False)
    train_loader_aug   = DataLoader(train_ds_aug,   batch_size=BATCH_SIZE, shuffle=True)
    train_loader_clean = DataLoader(train_ds_clean, batch_size=BATCH_SIZE, shuffle=False)

    val_loader = None
    if use_validation:
        val_ds     = RegressionDataset(X_val, Y_val, augment=False)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # [5] Model & optimizer
    model = Net(in_dim=14, hidden_dims=hidden_dims, out_dim=6, activation_str=activation).to(device)
    weights = torch.tensor(output_weights, dtype=torch.float32).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # [6] Training loop
    best_val_loss = float("inf")
    train_clean_losses, val_losses = [], []
    r2_train_clean, r2_val = [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader_aug:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = ((model(xb) - yb) ** 2 * weights).mean()
            loss.backward()
            optimizer.step()

        # compute metrics
        tr_loss = compute_loss(model, train_loader_clean, weights, device)
        train_clean_losses.append(tr_loss)
        # use compute_nth_r2 for the first output (index=0)
        r2_tr = compute_nth_r2(model, train_loader_clean, y_scaler, device, index=0)
        r2_train_clean.append(r2_tr)

        if use_validation:
            v_loss = compute_loss(model, val_loader, weights, device)
            val_losses.append(v_loss)
            r2_v = compute_nth_r2(model, val_loader, y_scaler, device, index=0)
            r2_val.append(r2_v)

        # save model if desired
        if save_artifacts:
            fname = f"PropLenght_model_{tag}.pth"
            if (not use_validation) or (use_validation and v_loss < best_val_loss):
                if use_validation:
                    best_val_loss = v_loss
                torch.save(model.state_dict(), fname)

        # logging (labels unchanged)
        if use_validation:
            print(f"Epoch {epoch:3d} ‖ train_loss: {tr_loss:.4f} ‖ "
                  f"val_loss: {v_loss:.4f} ‖ R²_1st: train={r2_tr:.4f}, val={r2_v:.4f}")
        else:
            print(f"Epoch {epoch:3d} ‖ train_loss: {tr_loss:.4f} ‖ R²_1st train={r2_tr:.4f}")

    # [7] Save parameters (models & params only)
    if save_artifacts:
        params = {
            "TRAIN_PATH": TRAIN_PATH,
            "BATCH_SIZE": BATCH_SIZE,
            "LR": LR,
            "NUM_EPOCHS": NUM_EPOCHS,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "NOISE_STD": NOISE_STD,
            "SEED": SEED,
            "hidden_dims": hidden_dims,
            "activation": activation,
            "output_weights": output_weights,
            "use_validation": use_validation,
            "validation_split": validation_split,
            "tag": tag
        }
        with open(f"PropLenght_params_{tag}.txt", "w") as f:
            for k, v in params.items():
                f.write(f"{k}: {v}\n")

    # [8] Plot training & validation curves (unchanged)
    epochs = np.arange(1, NUM_EPOCHS + 1)

    plt.figure()
    plt.plot(epochs, train_clean_losses, label="Train Loss (clean)")
    if use_validation:
        plt.plot(epochs, val_losses, label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Weighted MSE Loss")
    plt.title("Clean Train & Validation Loss")
    plt.legend(); plt.show()

    plt.figure()
    plt.plot(epochs, r2_train_clean, label="Train R² (1st)")
    if use_validation:
        plt.plot(epochs, r2_val, label="Val R² (1st)")
        plt.axhline(y=0.9, linestyle="--", label="0.9 Threshold")
    else:
        plt.axhline(y=0.9, linestyle="--", label="0.9 Threshold")
    plt.ylim(0, 1); plt.xlabel("Epoch"); plt.ylabel("R² Score")
    plt.title("First Output R² Over Epochs")
    plt.legend(); plt.show()
